# Практическая работа №1 - KNN

## Команда №1

Горюхин
Ковальский
Нахаев
Орлов

## Входные данные

На вход дан csv-файл с результатами опросника для определения вендора ОС мобильного телефона, загружаем его в виде pandas-датафрейма

In [6]:
import pandas as pd

df = pd.read_csv('knn-input.csv')
df.head()

,Отметка времени,"Как вы себя чувствуете, когда оказались в центре внимания в малознакомой компании?",Как чаще всего принимаются ваши повседневные решения о покупках?,"Покупая сложную технику или мебель, какой подход вам ближе?",Что вы выберете при покупке повседневной вещи?,Быстро ли твой телефон разряжается?,Ты часто снимаешь на телефон фото/видео?,"При выборе предмета интерьера или одежды, что станет решающим фактором?",Часто ли твой телефон тормозит/лагает?,"Тебе важно, чтобы все приложения со временем работали одинаково?",Любите ли вы путешествовать?,"Как часто ты меняешь смартфон на новый? Введите число лет (целое, например: 2)","Как часто вы играете в видеоигры? Введите примерное кол-во часов в неделю (целое, например: 5)",Часто ли вы публикуете фотографии или видео в социальных сетях?,Часто ли вы смотрите фильмы и сериалы на телефоне?,"Цвет глаз? (С большой буквы в именительном падеже единственное число, например: Зеленый)","Ближайшая станция метро? (Точное название, например: Преображенская площадь, Сокольники и т.п.)",У тебя айфон или андроид?
0,11.09.2026 20:01:26,Комфортно,Рационально,Чтобы всё сразу работало из коробки,Какой-то уникальный вариант,Да,Нет,"Дизайн, эстетика",Нет,Да,Да,4,5,Нет,Да,Зеленый,Юго-западная,Айфон
1,11.09.2026 20:01:27,Комфортно,Рационально,Чтобы всё сразу работало из коробки,Какой-то уникальный вариант,Да,Да,"Долговечность, польза",Нет,Да,Да,3,1,Да,Да,Карий,Бульвар Дмитрия Донского,Айфон
2,11.09.2026 20:01:38,Комфортно,Рационально,Чтобы всё сразу работало из коробки,Что-то популярное,Нет,Да,"Дизайн, эстетика",Нет,Да,Да,3,1,Нет,Нет,Карий,Братиславская,Айфон
3,11.09.2026 20:01:41,Дискомфортно,Рационально,Чтобы всё сразу работало из коробки,Какой-то уникальный вариант,Да,Нет,"Долговечность, польза",Нет,Да,Да,3,0,Нет,Нет,Серый,Авиамоторная,Айфон
4,11.09.2026 20:02:11,Комфортно,Рационально,Чтобы всё сразу работало из коробки,Что-то популярное,Да,Нет,"Долговечность, польза",Да,Да,Да,4,6,Нет,Нет,Зеленый,Волжская,Андроид


Нужно преобразовать входные данные в формат, подходящий для задач МО. По столбцам:
1. Как вы себя чувствуете, когда оказались в центре внимания в малознакомой компании? - **бинарный**
2. Как чаще всего принимаются ваши повседневные решения о покупках? - **бинарный**
3. Покупая сложную технику или мебель, какой подход вам ближе? - **бинарный**
4. Что вы выберете при покупке повседневной вещи? - **бинарный**
5. Быстро ли твой телефон разряжается? - **бинарный**
6. Ты часто снимаешь на телефон фото/видео? - **бинарный**
7. При выборе предмета интерьера или одежды, что станет решающим фактором? - **бинарный**
8. Часто ли твой телефон тормозит/лагает? - **бинарный**
9. Тебе важно, чтобы все приложения со временем работали одинаково? - **бинарный**
10. Любите ли вы путешествовать? - **бинарный**
11. Как часто ты меняешь смартфон на новый? - **целочисленный**
12. Как часто вы играете в видеоигры? - **целочисленный**
13. Часто ли вы публикуете фотографии или видео в социальных сетях? - **бинарный**
14. Часто ли вы смотрите фильмы и сериалы на телефоне? - **бинарный**
15. Цвет глаз? - **категориальный**
16. Ближайшая станция метро? - **категориальный**
17. У тебя айфон или андроид? - **бинарный** (целевой признак)

In [7]:
### preprocess df data

# drop timestamp
df = df.drop(columns=['Отметка времени'])

# binaries -> 0/1
binary_cols = list(df.columns[0:10]) + list(df.columns[12:14])
for col in binary_cols:
    values = df[col].astype(str).unique()
    if set(values) <= {'Да', 'Нет'}:
        df[col] = df[col].map({'Да': 1, 'Нет': 0})
    else:
        df[col] = df[col].map({values[0]: 0, values[1]: 1})

# integers
df[df.columns[10]] = df[df.columns[10]].astype(int)
df[df.columns[11]] = df[df.columns[11]].astype(int)

# target column
df[df.columns[16]] = df[df.columns[16]].map({'Айфон': 0, 'Андроид': 1})

# categorial + clean trash data a bit
eye, metro = df.columns[14], df.columns[15]
for col in [eye, metro]:
    df[col] = df[col].astype(str).str.strip().str.lower().str.replace('ё', 'е')

# synonyms
df[eye] = df[eye].replace({'карие': 'карий', 'карии': 'карий', 'коричневый': 'карий'})
df[eye] = df[eye].replace({'серо-зеленый': 'серый', 'серо-голубой': 'серый', 'серо-голубые': 'серый'})

# filter trash data
df = df[~df[eye].eq('кал') & ~df[metro].eq('0')]

df = pd.get_dummies(df, columns=[eye, metro], dtype=int)


In [3]:
df.columns

Index(['Как вы себя чувствуете, когда оказались в центре внимания в малознакомой компании?',
       'Как чаще всего принимаются ваши повседневные решения о покупках?',
       'Покупая сложную технику или мебель, какой подход вам ближе?',
       'Что вы выберете при покупке повседневной вещи?',
       'Быстро ли твой телефон разряжается?',
       'Ты часто снимаешь на телефон фото/видео?',
       'При выборе предмета интерьера или одежды, что станет решающим фактором?',
       'Часто ли твой телефон тормозит/лагает?',
       'Тебе важно, чтобы все приложения со временем работали одинаково?',
       'Любите ли вы путешествовать?',
       'Как часто ты меняешь смартфон на новый? Введите число лет (целое, например: 2)',
       'Как часто вы играете в видеоигры? Введите примерное кол-во часов в неделю (целое, например: 5)',
       'Часто ли вы публикуете фотографии или видео в социальных сетях?',
       'Часто ли вы смотрите фильмы и сериалы на телефоне?',
       'У тебя айфон или андроид?'

## Метод KNN

Теперь необходимо подобрать наилучшее k (число ближайших соседей, учитываемых при классификации)

In [8]:
# do KNN

import numpy as np

target = 'У тебя айфон или андроид?'
X = df.drop(columns=[target]).to_numpy(dtype=float)
y = df[target].to_numpy()

# split into train/test sets
rng = np.random.default_rng(42)
idx = rng.permutation(len(X))
cut = int(0.7 * len(X))
X_tr_raw, y_tr = X[idx[:cut]], y[idx[:cut]]
X_te_raw, y_te = X[idx[cut:]], y[idx[cut:]]

# z-score
mu = X_tr_raw.mean(axis=0)
sd = X_tr_raw.std(axis=0)
sd[sd == 0] = 1.0
X_tr = (X_tr_raw - mu) / sd
X_te = (X_te_raw - mu) / sd


def knn_predict(X_tr, y_tr, x, k):
    dists = np.sqrt(((X_tr - x) ** 2).sum(axis=1))
    k_nearest = np.argsort(dists)[:k]
    labels = y_tr[k_nearest]
    vote1 = (labels == 1).sum()
    if vote1 > k - vote1:
        return 1
    if vote1 < k - vote1:
        return 0
    return int(labels[0]) # tie case


def knn_accuracy(X_tr, y_tr, X_te, y_te, k):
    preds = [knn_predict(X_tr, y_tr, x, k) for x in X_te]
    return np.mean(np.array(preds) == y_te)

# k value tests
results = {k: knn_accuracy(X_tr, y_tr, X_te, y_te, k) for k in range(1, 16)}
best_k = max(results, key=results.get)

print('k\t|\taccuracy')
print('-'*50)
for k in range(1, 16):
    print(f'{k}\t|\t{results[k]:.3f}')
print('-'*50)

print(f'best k is {best_k} with accuracy of {results[best_k]:.3f}')

# compare test 'model' with baseline (mean values by class)
test_acc = knn_accuracy(X_tr, y_tr, X_te, y_te, best_k)
baseline = max((y_te == 1).mean(), (y_te == 0).mean())

print(f'test accuracy   = {test_acc:.3f}')
print(f'majority baseline = {baseline:.3f}')
print('модель лучше baseline' if test_acc > baseline else 'модель не лучше baseline')

k	|	accuracy
--------------------------------------------------
1	|	0.385
2	|	0.385
3	|	0.538
4	|	0.462
5	|	0.462
6	|	0.538
7	|	0.462
8	|	0.462
9	|	0.538
10	|	0.538
11	|	0.538
12	|	0.615
13	|	0.538
14	|	0.538
15	|	0.538
--------------------------------------------------
best k is 12 with accuracy of 0.615
test accuracy   = 0.615
majority baseline = 0.538
модель лучше baseline


### Предикт

Наибольшую точность предоставляет значение `k=12`, теперь можно проводить предикт

In [9]:
# try predict

# random input data
new_user = {
    'Как вы себя чувствуете, когда оказались в центре внимания в малознакомой компании?': 'Комфортно',
    'Как чаще всего принимаются ваши повседневные решения о покупках?': 'Рационально',
    'Покупая сложную технику или мебель, какой подход вам ближе?': 'Чтобы всё сразу работало из коробки',
    'Что вы выберете при покупке повседневной вещи?': 'Что-то популярное',
    'Быстро ли твой телефон разряжается?': 'Да',
    'Ты часто снимаешь на телефон фото/видео?': 'Нет',
    'При выборе предмета интерьера или одежды, что станет решающим фактором?': 'Долговечность, польза',
    'Часто ли твой телефон тормозит/лагает?': 'Нет',
    'Тебе важно, чтобы все приложения со временем работали одинаково?': 'Да',
    'Любите ли вы путешествовать?': 'Да',
    'Как часто ты меняешь смартфон на новый? Введите число лет (целое, например: 2)': 3,
    'Как часто вы играете в видеоигры? Введите примерное кол-во часов в неделю (целое, например: 5)': 5,
    'Часто ли вы публикуете фотографии или видео в социальных сетях?': 'Нет',
    'Часто ли вы смотрите фильмы и сериалы на телефоне?': 'Да',
    'Цвет глаз? (С большой буквы в именительном падеже единственное число, например: Зеленый)': 'Серый',
    'Ближайшая станция метро? (Точное название, например: Преображенская площадь, Сокольники и т.п.)': 'Сокольники',
}

raw = pd.read_csv('knn-input.csv')
feature_cols = list(raw.columns.drop(['Отметка времени', 'У тебя айфон или андроид?']))
inp = pd.DataFrame([new_user])[feature_cols]

# preprocessing
binary_cols = list(raw.columns[1:11]) + list(raw.columns[13:15])
for col in binary_cols:
    values = raw[col].astype(str).unique()
    if set(values) <= {'Да', 'Нет'}:
        inp[col] = inp[col].map({'Да': 1, 'Нет': 0})
    else:
        inp[col] = inp[col].map({values[0]: 0, values[1]: 1})

for col in [raw.columns[11], raw.columns[12]]:
    inp[col] = inp[col].astype(int)

eye, metro = raw.columns[15], raw.columns[16]
for col in [eye, metro]:
    inp[col] = inp[col].astype(str).str.strip().str.lower().str.replace('ё', 'е')
inp[eye] = inp[eye].replace({'карие': 'карий', 'карии': 'карий', 'коричневый': 'карий'})
inp[eye] = inp[eye].replace({'серо-зеленый': 'серый', 'серо-голубой': 'серый', 'серо-голубые': 'серый'})

# one-hot and reindex
X_cols = df.drop(columns=[target]).columns
inp = pd.get_dummies(inp, columns=[eye, metro], dtype=int)
inp = inp.reindex(columns=X_cols, fill_value=0)

# z-score
x = inp.to_numpy(dtype=float)[0]
x = (x - mu) / sd

# predict
pred = knn_predict(X_tr, y_tr, x, best_k)
print('Прогноз:', 'Андроид' if pred == 1 else 'Айфон', '| k =', best_k)

Прогноз: Андроид | k = 12
